# SAM3 Benchmarking on US30K Dataset

This notebook benchmarks the **Segment Anything Model 3 (SAM3)** on ultrasound images from the **US30K** dataset.

## Tracks:
1. **Automatic (AMG)**: Promptless mask generation.
2. **Text-Guided (TG)**: Using text prompts for specific object masks.
3. **Point Prompt**: Using a random point from the ground-truth mask as a prompt.

Configuration is managed via `config.yaml`.

In [ ]:
import os
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import random

# Add sam3 to path if needed
import sys
sys.path.append(os.path.abspath('sam3'))

import sam3
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

In [ ]:
# 1. Load Configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

dataset_root = Path(config['paths']['dataset_root'])
weights_dir = Path(config['paths']['weights_dir'])
output_dir = Path(config['paths']['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

track_mode = config['benchmark']['track_mode']
device = config['model']['device']

print(f"Running in {track_mode} mode on {device}.")

## 2. Weight Download (if needed)

SAM3 weights should be in the `checkpoints` directory. We use `huggingface_hub` to ensure they are available.

In [ ]:
from huggingface_hub import snapshot_download

if not weights_dir.exists() or len(list(weights_dir.glob('*.pt'))) == 0:
    print("Downloading SAM3 weights...")
    snapshot_download(
        repo_id=config['model']['sam3_model_id'],
        local_dir=weights_dir,
    )
else:
    print(f"Weights already present in {weights_dir}")

## 3. Initialize Model

In [ ]:
sam3_root = os.path.abspath('sam3')
bpe_path = os.path.join(sam3_root, "assets/bpe_simple_vocab_16e6.txt.gz")

# Build the model
model = build_sam3_image_model(bpe_path=bpe_path, enable_inst_interactivity=True).to(device)
processor = Sam3Processor(model, confidence_threshold=config['model']['confidence_threshold'])

print("SAM3 initialized successfully.")

## 4. Benchmarking Loop

We iterate through the specified subfolders and perform segmentation based on the chosen track.

In [ ]:
from monai_wg.metrics import MonaiMetricWrapper
import pandas as pd

# Initialize metrics
metrics_wrapper = MonaiMetricWrapper(num_classes=2, include_background=False)
all_results = []

for subfolder in config['paths']['subfolders']:
    img_dir = dataset_root / subfolder / "img"
    label_dir = dataset_root / subfolder / "label"
    
    if not img_dir.exists():
        print(f"Skipping {subfolder}: img directory not found.")
        continue
        
    img_files = sorted(list(img_dir.glob("*.png")))[:config['benchmark']['num_images_per_folder']]
    
    print(f"Benchmarking {subfolder} ({len(img_files)} images)...")
    
    for img_path in tqdm(img_files):
        # Load Image
        image_pil = Image.open(img_path).convert("RGB")
        image_np = np.array(image_pil)
        
        # Load Label (if exists)
        label_path = label_dir / img_path.name
        if label_path.exists():
            label_np = np.array(Image.open(label_path).convert("L"))
            label_np = (label_np > 0).astype(np.uint8)
        else:
            label_np = None
            
        # 4.1 Inference
        with torch.no_grad():
            inference_state = processor.set_image(image_pil)
            
            if track_mode == 'text_prompt':
                output = processor.set_text_prompt(
                    state=inference_state, 
                    prompt=config['benchmark']['text_prompt']
                )
                pred_mask = output['masks'][0] if len(output['masks']) > 0 else np.zeros(image_np.shape[:2], dtype=np.uint8)
            
            elif track_mode == 'point_prompt':
                if label_np is not None and np.any(label_np > 0):
                    # Find positive pixels
                    y_coords, x_coords = np.where(label_np > 0)
                    idx = random.randint(0, len(y_coords) - 1)
                    input_point = np.array([[x_coords[idx], y_coords[idx]]])
                    input_label = np.array([1])
                    
                    # Use predict_inst for point prompts
                    masks, scores, _ = model.predict_inst(
                        inference_state,
                        point_coords=input_point,
                        point_labels=input_label,
                        multimask_output=True,
                    )
                    # Select best mask per score
                    pred_mask = masks[np.argmax(scores)]
                else:
                    pred_mask = np.zeros(image_np.shape[:2], dtype=np.uint8)
                    
            elif track_mode == 'automatic':
                # Fallback to generic prompt if AMG class not found
                output = processor.set_text_prompt(state=inference_state, prompt="object")
                pred_mask = (np.mean(output['masks'], axis=0) > 0.5).astype(np.uint8) if len(output['masks']) > 0 else np.zeros(image_np.shape[:2], dtype=np.uint8)
        
        # 4.2 Calculate Metrics
        if label_np is not None:
            metrics_wrapper.update(pred_mask[None, None, ...], label_np[None, None, ...])
            res = metrics_wrapper.compute()
            res['image'] = img_path.name
            res['dataset'] = subfolder
            res['model'] = track_mode
            all_results.append(res)
            metrics_wrapper.reset()

df_results = pd.DataFrame(all_results)
df_results.to_csv(output_dir / f"sam3_benchmark_{track_mode}.csv", index=False)
print(f"Benchmark completed. Results saved to {output_dir}")

## 5. Visualizations

In [ ]:
if not df_results.empty:
    print("Summary Metrics:")
    summary = df_results.drop(columns=['image', 'dataset', 'model'], errors='ignore').mean()
    print(summary)
    
    from monai_wg.plotting import plot_metric_distribution, plot_radar_chart, plot_summary_report, plot_segmentation
    
    # 1. Plot continuous metric distribution
    plot_metric_distribution(df_results, save_path=output_dir / f"metric_distribution_{track_mode}.png")
    
    # 2. Plot Radar Chart (comparing performance across multiple metrics)
    summary_df = summary.to_frame(name=track_mode).T
    summary_df.index.name = 'Model'
    plot_radar_chart(summary_df, save_path=output_dir / f"radar_chart_{track_mode}.png")
    
    # 3. Plot Summary Report (using the last processed image as an example)
    if 'image_np' in locals() and 'pred_mask' in locals() and 'label_np' in locals() and label_np is not None:
        plot_summary_report(
            metrics_df=df_results,
            overlay_info={
                'image': image_np,
                'label': label_np,
                'pred': pred_mask
            },
            save_path=output_dir / f"summary_report_{track_mode}.png"
        )
        
        # 4. Save individual sample overlay
        plot_segmentation(
            image=image_np, 
            label=label_np,
            pred=pred_mask, 
            title=f"Example Segmentation: {track_mode}",
            save_path=output_dir / f"example_segmentation_{track_mode}.png"
        )
